# llamacpp-chat2（Google Colab）セットアップ

この Notebook は **llamacpp-chat2** の Colab GPU 向け手順です。

1. Google Drive をマウントし、リポジトリを配置
2. `llama-server` をビルド（CUDA。Drive 上にキャッシュ）
3. 一時領域 `/content` に venv を作成し、制御API + Gradio を起動

| 用途 | パス | 寿命 |
|------|------|------|
| リポジトリ | `/content/drive/MyDrive/llamacpp-chat2` | Drive 永続 |
| llama.cpp | `.../vendor/llama.cpp` | Drive 永続 |
| venv / モデル | `/content/llamacpp-chat2` | ランタイム再起動で消失 |

## 事前準備

**ランタイム → ランタイムのタイプを変更 → ハードウェア アクセラレータ → GPU**

Public URL（Gradio share）が表示されたら手元ブラウザで開いてください。


## GPU 確認

In [ ]:
!nvidia-smi

## Drive マウントとパス設定

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/mckey-dev/llamacpp-chat2.git"
REPO = Path("/content/drive/MyDrive/llamacpp-chat2")
RUNTIME = Path("/content/llamacpp-chat2")
MODELS_DIR = RUNTIME
VENV_SERVER = RUNTIME / ".venv-server"
VENV_FRONTEND = RUNTIME / ".venv-frontend"
PY_SERVER = VENV_SERVER / "bin" / "python"
PY_FRONTEND = VENV_FRONTEND / "bin" / "python"
CONTROL_TOKEN = "llamacpp-chat2"

RUNTIME.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("REPO (永続):", REPO)
print("RUNTIME (一時):", RUNTIME)
print("MODELS_DIR:", MODELS_DIR)

## リポジトリクローン / 更新

In [ ]:
import subprocess

if not REPO.is_dir():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
    print("クローンしました:", REPO)
else:
    print("リポジトリは既に存在します:", REPO)
    try:
        subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only"])
    except subprocess.CalledProcessError:
        print("git pull に失敗しました。手動で更新してください。")

for name in ("server/control_api.py", "ui/app.py", "requirements-frontend.txt"):
    assert (REPO / name).is_file(), f"必須ファイルがありません: {REPO / name}"
print("リポジトリ OK")

## ビルドツールと llama-server

先に **Drive マウントとパス設定** セルを実行してください。

- ビルドは `/content/llama.cpp` で行い、完成バイナリを Drive にコピーします
- メモリ不足で落ちる場合はセル先頭の `BUILD_JOBS = 1` にして再実行（増分ビルド）
- CUDA が使えない場合は `BUILD_WITH_CUDA = False`


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

# True: CUDA（GPU ランタイム）、False: CPU のみ
BUILD_WITH_CUDA = False
# Colab はメモリが少ないので並列を抑える（足りなければ 1）
BUILD_JOBS = 2
# True で build/ を消して最初から
FORCE_CLEAN_BUILD = False

if "REPO" not in globals():
    REPO = Path("/content/drive/MyDrive/llamacpp-chat2")
    print("REPO が未定義だったため再設定しました:", REPO)

assert REPO.is_dir(), (
    f"リポジトリがありません: {REPO}\n"
    "先に「Drive マウントとパス設定」と「リポジトリクローン」セルを実行してください。"
)


def _run(cmd, *, cwd=None, env=None) -> None:
    """コマンドを実行し、出力を Notebook に逐次表示する。"""
    print("$", " ".join(str(c) for c in cmd), flush=True)
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"終了コード {code}: {' '.join(str(c) for c in cmd)}")


def _find_nvcc() -> Path | None:
    for p in (
        Path("/usr/local/cuda/bin/nvcc"),
        Path("/usr/bin/nvcc"),
    ):
        if p.is_file():
            return p
    which = shutil.which("nvcc")
    return Path(which) if which else None


_run(["apt-get", "update"])
_run(
    [
        "apt-get",
        "install",
        "-y",
        "cmake",
        "build-essential",
        "git",
        "libcurl4-openssl-dev",
    ]
)

BUILD_SRC = Path("/content/llama.cpp")
PERSIST_BIN = (
    REPO / "vendor" / "llama.cpp" / "build" / "bin" / "llama-server"
)
LOCAL_BIN = BUILD_SRC / "build" / "bin" / "llama-server"

if PERSIST_BIN.is_file():
    LLAMA_SERVER = PERSIST_BIN
    print("既存の llama-server を使います:", LLAMA_SERVER)
elif LOCAL_BIN.is_file():
    LLAMA_SERVER = LOCAL_BIN
    print("ローカルの llama-server を使います:", LLAMA_SERVER)
else:
    if not BUILD_SRC.is_dir():
        print("llama.cpp を /content にクローンします…")
        _run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/ggml-org/llama.cpp.git",
                str(BUILD_SRC),
            ]
        )

    env = os.environ.copy()
    cmake_args = ["cmake", "-B", "build", "-DCMAKE_BUILD_TYPE=Release"]

    if BUILD_WITH_CUDA:
        nvcc = _find_nvcc()
        if nvcc is None:
            raise RuntimeError(
                "nvcc が見つかりません。ランタイムを GPU にしてから再実行するか、\n"
                "セル先頭で BUILD_WITH_CUDA = False にして CPU ビルドしてください。"
            )
        cuda_home = str(nvcc.parent.parent)
        env["CUDA_HOME"] = cuda_home
        env["CUDA_PATH"] = cuda_home
        env["CUDACXX"] = str(nvcc)
        env["PATH"] = f"{nvcc.parent}:{env.get('PATH', '')}"
        cmake_args.extend(
            [
                "-DGGML_CUDA=ON",
                f"-DCMAKE_CUDA_COMPILER={nvcc}",
            ]
        )
        print("CUDA ビルド:", cuda_home, "nvcc=", nvcc)
    else:
        cmake_args.append("-DGGML_CUDA=OFF")
        print("CPU ビルド（推論は遅くなります）")

    build_dir = BUILD_SRC / "build"
    cache = build_dir / "CMakeCache.txt"
    if FORCE_CLEAN_BUILD and build_dir.is_dir():
        print("FORCE_CLEAN_BUILD: 既存 build/ を削除します…")
        shutil.rmtree(build_dir)
    elif build_dir.is_dir() and not cache.is_file():
        print("不完全な build/ を削除します…")
        shutil.rmtree(build_dir)

    if not (build_dir / "CMakeCache.txt").is_file():
        print("cmake configure…")
        _run(cmake_args, cwd=str(BUILD_SRC), env=env)
    else:
        print("既存の CMake キャッシュを使います（増分ビルド）")

    print(f"cmake build（-j{BUILD_JOBS}。メモリ不足なら BUILD_JOBS=1）…")
    _run(
        [
            "cmake",
            "--build",
            "build",
            "--config",
            "Release",
            "-j",
            str(BUILD_JOBS),
        ],
        cwd=str(BUILD_SRC),
        env=env,
    )

    assert LOCAL_BIN.is_file(), f"ビルド後もバイナリがありません: {LOCAL_BIN}"
    PERSIST_BIN.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(LOCAL_BIN, PERSIST_BIN)
    LLAMA_SERVER = PERSIST_BIN
    print("Drive にコピーしました:", LLAMA_SERVER)

assert LLAMA_SERVER.is_file(), f"llama-server がありません: {LLAMA_SERVER}"
print("llama-server:", LLAMA_SERVER)


## Python venv とフロント依存

Gradio 4.40 は **Python 3.10–3.13** が必要です。セルは利用可能な 3.10–3.13 を自動選択し、必要なら `python3.X-venv` を入れます。


In [ ]:
def _stream(cmd, *, cwd=None, env=None):
    """サブプロセスを実行し、出力を Notebook に逐次表示する。"""
    import subprocess

    print("$", " ".join(str(c) for c in cmd))
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"終了コード {code}: {' '.join(str(c) for c in cmd)}")


def _pick_python() -> tuple[str, str]:
    """Gradio 4.40 向けに Python 3.10-3.13 を選ぶ。

    Returns
    -------
    path, version
        実行ファイルパスと ``X.Y`` 文字列。
    """
    import shutil
    import subprocess
    import sys

    candidates = [
        "python3.11",
        "python3.10",
        "python3.12",
        "python3.13",
        "python3",
        sys.executable,
    ]
    for name in candidates:
        path = shutil.which(name) if name != sys.executable else name
        if not path:
            continue
        try:
            out = subprocess.check_output(
                [
                    path,
                    "-c",
                    "import sys; print(f'{sys.version_info[0]}.{sys.version_info[1]}')",
                ],
                text=True,
            ).strip()
            major, minor = map(int, out.split("."))
        except (subprocess.CalledProcessError, ValueError, OSError):
            continue
        if (major, minor) >= (3, 14):
            print(f"スキップ（3.14+ 非対応）: {path} ({out})")
            continue
        if (major, minor) < (3, 10):
            print(f"スキップ（3.10 未満）: {path} ({out})")
            continue
        print(f"venv 用 Python: {path} ({out})")
        return path, out
    raise RuntimeError(
        "Python 3.10-3.13 が見つかりません。"
        " 例: apt-get install -y python3.11 python3.11-venv"
    )


def _ensure_venv_package(py_path: str, py_version: str) -> None:
    """ensurepip 用に pythonX.Y-venv を入れる。"""
    import shutil
    import subprocess

    pkg = f"python{py_version}-venv"
    probe = subprocess.run(
        [py_path, "-c", "import ensurepip"],
        capture_output=True,
        text=True,
    )
    if probe.returncode == 0:
        print(f"ensurepip OK ({py_path})")
        return

    print(f"{pkg} をインストールします…")
    apt = shutil.which("apt-get")
    if not apt:
        raise RuntimeError(
            f"ensurepip が使えません。手動で {pkg} を入れてください。"
        )
    _stream(["apt-get", "update"])
    try:
        _stream(["apt-get", "install", "-y", pkg])
    except RuntimeError:
        _stream(["sudo", "apt-get", "update"])
        _stream(["sudo", "apt-get", "install", "-y", pkg])


def _cleanup_broken_venvs() -> None:
    """不完全・非対応バージョンの venv を削除する。"""
    import shutil
    import subprocess

    for venv_dir, label in (
        (VENV_FRONTEND, ".venv-frontend"),
        (VENV_SERVER, ".venv-server"),
    ):
        if not venv_dir.exists():
            continue
        py = venv_dir / "bin" / "python"
        if not py.is_file():
            print(f"{label}: 不完全なため削除します…")
            shutil.rmtree(venv_dir, ignore_errors=True)
            continue
        try:
            ver = subprocess.check_output(
                [
                    str(py),
                    "-c",
                    "import sys; print(f'{sys.version_info[0]}.{sys.version_info[1]}')",
                ],
                text=True,
            ).strip()
            major, minor = map(int, ver.split("."))
            # pip が無ければ ensurepip 失敗の残骸
            pip_ok = subprocess.run(
                [str(py), "-m", "pip", "--version"],
                capture_output=True,
            ).returncode == 0
            if (major, minor) >= (3, 14) or not pip_ok:
                print(f"{label}: 破損または非対応 (Python {ver}) のため削除します…")
                shutil.rmtree(venv_dir, ignore_errors=True)
        except (subprocess.CalledProcessError, ValueError, OSError):
            print(f"{label}: 検査失敗のため削除します…")
            shutil.rmtree(venv_dir, ignore_errors=True)


import shutil
import subprocess
from pathlib import Path

assert "VENV_SERVER" in globals() and "VENV_FRONTEND" in globals(), (
    "先にパス設定セルを実行してください"
)

PY_BASE, PY_VER = _pick_python()
_ensure_venv_package(PY_BASE, PY_VER)
_cleanup_broken_venvs()

if not PY_SERVER.is_file():
    print("Creating .venv-server …")
    subprocess.check_call([PY_BASE, "-m", "venv", str(VENV_SERVER)])

if not PY_FRONTEND.is_file():
    print("Creating .venv-frontend …")
    subprocess.check_call([PY_BASE, "-m", "venv", str(VENV_FRONTEND)])

_stream(
    [str(PY_FRONTEND), "-m", "pip", "install", "--upgrade", "pip"],
    cwd=str(REPO),
)
_stream(
    [
        str(PY_FRONTEND),
        "-m",
        "pip",
        "install",
        "-r",
        str(REPO / "requirements-frontend.txt"),
    ],
    cwd=str(REPO),
)
print("venv-server:", PY_SERVER)
print("venv-frontend:", PY_FRONTEND)
_stream([str(PY_FRONTEND), "--version"])


## 環境変数

In [ ]:
import os
from pathlib import Path

MODELS_DIR.mkdir(parents=True, exist_ok=True)

if "LLAMA_SERVER" not in globals():
    candidates = [
        REPO / "vendor" / "llama.cpp" / "build" / "bin" / "llama-server",
        Path("/content/llama.cpp/build/bin/llama-server"),
    ]
    for c in candidates:
        if c.is_file():
            LLAMA_SERVER = c
            break
    else:
        raise FileNotFoundError(
            "llama-server が見つかりません。先にビルドセルを実行してください。"
        )

os.environ["LLAMACPP_CHAT2_ROOT"] = str(REPO)
os.environ["LLAMACPP_CHAT2_MODELS_DIR"] = str(MODELS_DIR)
os.environ["LLAMACPP_CHAT2_LLAMA_SERVER"] = str(LLAMA_SERVER)
os.environ["LLAMACPP_CHAT2_CONTROL_TOKEN"] = CONTROL_TOKEN
os.environ["LLAMACPP_CHAT2_LLAMA_HOST"] = "127.0.0.1"
os.environ["LLAMACPP_CHAT2_LLAMA_PORT"] = "8080"
os.environ["LLAMACPP_CHAT2_CONTROL_HOST"] = "127.0.0.1"
os.environ["LLAMACPP_CHAT2_CONTROL_PORT"] = "8090"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["GRADIO_SHARE"] = "True"
os.environ["GRADIO_SERVER_NAME"] = "0.0.0.0"
os.environ["MPLBACKEND"] = "Agg"

print("MODELS_DIR:", MODELS_DIR)
print("LLAMA_SERVER:", LLAMA_SERVER)
print("制御API: http://127.0.0.1:8090")
print("推論:    http://127.0.0.1:8080")


## 制御API 起動（バックグラウンド）

セル実行で起動します。**停止**は下の「制御APIを停止」ボタンを押してください（`llama-server` も unload してから制御APIを止めます）。

ボタンが出ない場合は、同じセル内の `STOP_CONTROL_API = True` にして再実行してください。


In [ ]:
import os
import socket
import subprocess
import time
import urllib.request

# ボタンが使えないとき: True にしてこのセルを再実行
STOP_CONTROL_API = False


def _port_open(host: str, port: int) -> bool:
    try:
        with socket.create_connection((host, port), timeout=0.5):
            return True
    except OSError:
        return False


def stop_control_api() -> None:
    """llama-server を unload し、制御APIプロセスを停止する。"""
    token = os.environ.get("LLAMACPP_CHAT2_CONTROL_TOKEN", "llamacpp-chat2")
    try:
        req = urllib.request.Request(
            "http://127.0.0.1:8090/v1/control/unload",
            data=b"{}",
            headers={
                "Content-Type": "application/json",
                "X-Control-Token": token,
            },
            method="POST",
        )
        with urllib.request.urlopen(req, timeout=10) as resp:
            resp.read()
        print("llama-server を unload しました")
    except Exception as e:
        print("unload スキップ:", e)

    proc = globals().get("CONTROL_PROC")
    if proc is not None and proc.poll() is None:
        pid = proc.pid
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=5)
        print(f"制御APIを停止しました (pid={pid})")
    else:
        print("CONTROL_PROC はありません（カーネル再起動後など）")
    globals()["CONTROL_PROC"] = None

    # 孤児プロセス対策（ポートが残っている場合）
    if _port_open("127.0.0.1", 8090):
        print("8090 がまだ開いているため、control_api プロセスを探して停止します…")
        try:
            subprocess.run(
                ["pkill", "-f", "server.control_api"],
                check=False,
                timeout=10,
            )
        except Exception as e:
            print("pkill スキップ:", e)
        time.sleep(0.5)
        if _port_open("127.0.0.1", 8090):
            print("警告: 8090 がまだ応答しています。マシン再起動か手動でプロセスを終了してください。")
        else:
            print("ポート 8090 を解放しました")
    else:
        print("制御APIは停止済みです (8090 closed)")


if STOP_CONTROL_API:
    stop_control_api()
elif "CONTROL_PROC" in globals() and CONTROL_PROC and CONTROL_PROC.poll() is None:
    print("制御APIは既に起動しています (pid=", CONTROL_PROC.pid, ")")
else:
    env = os.environ.copy()
    CONTROL_PROC = subprocess.Popen(
        [str(PY_SERVER), "-m", "server.control_api"],
        cwd=str(REPO),
        env=env,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
    )
    print("制御APIを起動しました pid=", CONTROL_PROC.pid)

    ok = False
    for _ in range(60):
        if CONTROL_PROC.poll() is not None:
            raise RuntimeError(
                f"制御APIが終了しました exit_code={CONTROL_PROC.returncode}"
            )
        try:
            with socket.create_connection(("127.0.0.1", 8090), timeout=1):
                ok = True
                break
        except OSError:
            time.sleep(0.5)
    if not ok:
        raise RuntimeError("制御API (8090) が応答しません")
    print("制御API ready: http://127.0.0.1:8090")

_widgets_ok = False
try:
    import ipywidgets as widgets
    from IPython.display import display

    btn_stop = widgets.Button(
        description="制御APIを停止",
        button_style="danger",
        layout=widgets.Layout(width="auto"),
    )

    def _on_stop(_btn) -> None:
        stop_control_api()

    btn_stop.on_click(_on_stop)
    display(btn_stop)
    _widgets_ok = True
except Exception as e:
    print("停止ボタンを表示できません（ipywidgets 非対応の可能性）:", e)
    print("代わりに STOP_CONTROL_API = True にして再実行してください。")

if not _widgets_ok:
    print("ヒント: STOP_CONTROL_API = True で停止できます。")


## Gradio UI 起動

**Public URL** が表示されたら開いてください。

ランタイム再起動後は venv 作成以降のセルを再実行してください（Drive 上のリポジトリと llama.cpp ビルドは残ります）。

停止: **ランタイム → 実行を中断**

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

_DROP_PREFIXES = (
    "clip_model_loader: tensor[",
    "add_text:",
    "add_media:",
    "tokenize:",
    "find_slot:",
    "image_tokens->",
    "batch_f32",
    "load_hparams:",
)
_KEEP_SNIPPETS = (
    "Running on",
    "public URL",
    "Public URL",
    "Gradio",
    "Error",
    "Traceback",
    "Exception",
    "エラー",
    "control API",
)


def _notebook_should_print(line: str) -> bool:
    s = line.strip()
    if not s:
        return False
    if any(k in s for k in _KEEP_SNIPPETS):
        return True
    if any(s.startswith(p) for p in _DROP_PREFIXES):
        return False
    return True

os.chdir(REPO)
env = os.environ.copy()
env["MPLBACKEND"] = "Agg"


print("Gradio UI を起動します…")
print("Connection タブの URL は既定のまま (8080 / 8090) で動作します。")
print("1. Connection: モデル追加 → 一覧更新 → Load")
print("2. Chat: 会話（画像添付可）")
print("停止: このセルを Interrupt\n")

proc = subprocess.Popen(
    [str(PY_FRONTEND), "-m", "ui.app"],
    cwd=str(REPO),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
try:
    assert proc.stdout is not None
    for line in proc.stdout:
        if _notebook_should_print(line):
            print(line, end="")
            sys.stdout.flush()
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"ui.app が終了コード {code} で終了しました")
except KeyboardInterrupt:
    proc.terminate()
    try:
        proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        proc.kill()
    if "stop_control_api" in globals():
        stop_control_api()
    elif "CONTROL_PROC" in globals() and CONTROL_PROC:
        CONTROL_PROC.terminate()
    print("\n停止しました")


## 仮想環境の削除

依存の入れ直しや破損した venv を消したいときに使います。**パス設定セルを先に実行**してください。

- `.venv-server` … 制御API用
- `.venv-frontend` … Gradio UI 用

ボタンを押すと、その下のセル出力に結果が出ます。  
（Paperspace では `Output` ウィジェットが使えないことがあるため、結果は通常の print で表示します。）

削除後は「Python venv とフロント依存」セルを再実行してください。

ボタンが表示されない場合は、同じセル内の `DELETE_VENV_*` フラグを編集してセルを再実行してください。


In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

assert "VENV_SERVER" in globals() and "VENV_FRONTEND" in globals(), (
    "先にパス設定セルを実行してください（VENV_SERVER / VENV_FRONTEND）"
)


def _stop_control_api() -> None:
    """制御API 子プロセスがあれば停止する。"""
    fn = globals().get("stop_control_api")
    if callable(fn):
        fn()
        return
    proc = globals().get("CONTROL_PROC")
    if proc is None:
        return
    if proc.poll() is not None:
        return
    proc.terminate()
    try:
        proc.wait(timeout=10)
    except Exception:
        proc.kill()
    print("制御APIを停止しました")


def _rm_venv(path: Path, label: str) -> None:
    if not path.exists():
        print(f"{label}: 存在しません（{path}）")
        return
    shutil.rmtree(path)
    print(f"{label}: 削除しました → {path}")


def delete_venvs(*, server: bool, frontend: bool) -> None:
    """指定した venv を削除する。"""
    print("対象:")
    print(f"  .venv-server   : {VENV_SERVER}  存在={VENV_SERVER.is_dir()}")
    print(f"  .venv-frontend : {VENV_FRONTEND}  存在={VENV_FRONTEND.is_dir()}")
    if server:
        _stop_control_api()
        _rm_venv(VENV_SERVER, ".venv-server")
    if frontend:
        _rm_venv(VENV_FRONTEND, ".venv-frontend")
    if not server and not frontend:
        print("削除対象がありません。")
    else:
        print("完了。必要なら「Python venv とフロント依存」セルを再実行してください。")


# ボタンが使えない環境向けフォールバック（セル実行時に使う）
DELETE_VENV_SERVER = False
DELETE_VENV_FRONTEND = False

_widgets_ok = False
try:
    import ipywidgets as widgets
    from IPython.display import display

    btn_server = widgets.Button(
        description=".venv-server を削除",
        button_style="warning",
        layout=widgets.Layout(width="auto"),
    )
    btn_frontend = widgets.Button(
        description=".venv-frontend を削除",
        button_style="warning",
        layout=widgets.Layout(width="auto"),
    )
    btn_both = widgets.Button(
        description="両方削除",
        button_style="danger",
        layout=widgets.Layout(width="auto"),
    )

    btn_server.on_click(lambda _b: delete_venvs(server=True, frontend=False))
    btn_frontend.on_click(lambda _b: delete_venvs(server=False, frontend=True))
    btn_both.on_click(lambda _b: delete_venvs(server=True, frontend=True))

    print("対象:")
    print(f"  .venv-server   : {VENV_SERVER}  存在={VENV_SERVER.is_dir()}")
    print(f"  .venv-frontend : {VENV_FRONTEND}  存在={VENV_FRONTEND.is_dir()}")
    display(widgets.HBox([btn_server, btn_frontend, btn_both]))
    _widgets_ok = True
except Exception as e:
    print("ボタン UI を表示できません（ipywidgets 非対応の可能性）:", e)
    print("代わりに DELETE_VENV_SERVER / DELETE_VENV_FRONTEND を True にして再実行してください。")

if not _widgets_ok and (DELETE_VENV_SERVER or DELETE_VENV_FRONTEND):
    delete_venvs(server=DELETE_VENV_SERVER, frontend=DELETE_VENV_FRONTEND)
